# Ark+ on 4 MedMNIST datasets: smoke test, full run, results

One Ark+ model (Swin-Base, 224x224) trained jointly on **ChestMNIST, DermaMNIST, RetinaMNIST, BreastMNIST**
with the official Ark+ pretraining code. The code lives on GitHub; this notebook only clones it, runs it and plots the results.

| Step | What happens |
|---|---|
| 1. Settings | choose `smoke` or `full`, repo URL, where files go |
| 2. Setup | clone the repo, install packages, show the GPU |
| 3. Data | download MedMNIST+ 224 and convert to memory-mapped `.npy` |
| 4. GPU probe | find the largest micro-batch that fits, estimate run time |
| 5. Smoke test | 2 tiny epochs through the full pipeline (train, EMA, val, early stop, resume, test) |
| 6. Full run | 50 epochs, early stopping patience 5, resumes automatically |
| 7. Results | table + charts against your individual baselines |

Everything that differs from the released code is listed in `Ark_Plus/Pretraining_MedMNIST/PATCHES.md`.

## 1. Settings

In [ ]:
import os, sys, json, glob, shutil, subprocess, time
from pathlib import Path

# ---- what to run ------------------------------------------------------------
RUN_MODE      = "smoke"   # "smoke" first; switch to "full" once the smoke test passes
EXP_NAME      = "medmnist4"
EPOCHS        = 50
PATIENCE      = 5         # early stopping: stop after 5 epochs without val improvement
TEST_EPOCH    = 10        # periodic test as in the original (0 = only the final test)
EFFECTIVE_BATCH = 200     # README Swin-Base value; kept via gradient accumulation
USE_AMP       = True      # float16 mixed precision (needed on 8 GB)

# ---- code on GitHub ---------------------------------------------------------
REPO_URL = "https://github.com/ArnabmoyPaul/Ark.git"   # your fork of jlianglab/Ark
BRANCH   = "main"

# ---- where things go (auto-detected) ----------------------------------------
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB  = "google.colab" in sys.modules
if ON_KAGGLE:
    WORK = Path("/kaggle/working");  DATA = Path("/tmp/medmnist")          # data outside /kaggle/working (not saved)
elif ON_COLAB:
    WORK = Path("/content");         DATA = Path("/content/medmnist")
else:
    WORK = Path.cwd() / "ark_work";  DATA = WORK / "medmnist"
REPO_DIR = WORK / "Ark"
CODE_DIR = REPO_DIR / "Ark_Plus" / "Pretraining_MedMNIST"
RUNS     = WORK / "runs"
NPZ_DIR  = DATA / "npz"             # put *_224.npz here if automatic download is blocked

# Kaggle: to continue a run from a previous session, add that notebook version's
# output as an input and point this at its runs folder, e.g. "/kaggle/input/ark-medmnist/runs"
RESUME_FROM = ""
TIME_LIMIT_H = 11.3 if ON_KAGGLE else 0   # clean stop before Kaggle's 12 h limit

WORKERS = 4 if (ON_KAGGLE or ON_COLAB) else min(4, os.cpu_count() or 2)
DATASETS = ["ChestMNIST", "DermaMNIST", "RetinaMNIST", "BreastMNIST"]
SWIN_B_URL = "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22kto1k.pth"
for p in [WORK, DATA, RUNS]:
    p.mkdir(parents=True, exist_ok=True)
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local", "| work:", WORK, "| data:", DATA)

## 2. Setup: clone the code and install packages

In [ ]:
def sh(cmd, cwd=None, check=True):
    """Run a command and stream its output live into the notebook."""
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    p = subprocess.Popen(cmd, cwd=cwd, shell=isinstance(cmd, str), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

if (REPO_DIR / ".git").exists():
    sh(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
elif not REPO_DIR.exists():
    sh(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
assert CODE_DIR.exists(), f"{CODE_DIR} not found: did you push Ark_Plus/Pretraining_MedMNIST to your fork?"

sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements_medmnist.txt")])

import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        pr = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {pr.name}, {pr.total_memory/2**30:.1f} GB")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print("No GPU visible: the smoke test will run on CPU (slow). Full training needs a GPU.")

sh([sys.executable, "tests/test_accumulation.py"], cwd=CODE_DIR)   # accumulation == full batch, BN-EMA works

## 3. Data: MedMNIST+ 224x224

In [ ]:
sh([sys.executable, "prepare_medmnist.py", "--data_root", str(DATA), "--npz_dir", str(NPZ_DIR),
    "--datasets", "chestmnist", "dermamnist", "retinamnist", "breastmnist"], cwd=CODE_DIR)

import numpy as np
for d in DATASETS:
    n = {s: np.load(DATA / d.lower() / f"{s}_labels.npy").shape[0] for s in ["train", "val", "test"]}
    print(f"{d:<12s} train {n['train']:>6d}  val {n['val']:>6d}  test {n['test']:>6d}")

## 4. GPU probe: micro-batch and time estimate

Tries micro-batches that divide 200 until one fits (student forward+backward, teacher forward, SGD step),
then measures speed. Effective batch stays 200 through gradient accumulation, so the optimisation is the same as batch 200.

In [ ]:
if DEVICE == "cuda":
    sh([sys.executable, "probe_gpu.py", "--effective_batch", str(EFFECTIVE_BATCH), "--amp", str(USE_AMP),
        "--epochs", str(EPOCHS), "--test_epoch", str(TEST_EPOCH)], cwd=CODE_DIR)
    probe = json.load(open(CODE_DIR / "probe_result.json"))
    MICRO, ACCUM = probe["micro_batch"], probe["accum_steps"]
    print(f"\nProjected full run: about {probe[f'projected_hours_{EPOCHS}_epochs']} h "
          f"(+10-30% for data loading; early stopping can cut this)")
else:
    MICRO, ACCUM = 2, EFFECTIVE_BATCH // 2
print("micro-batch", MICRO, "x accum", ACCUM, "=", MICRO * ACCUM)

## 5. The training command

In [ ]:
def ark_cmd(datasets, exp_name, epochs, extra=()):
    cmd = [sys.executable, "main_ark.py"]
    for d in datasets:
        cmd += ["--data_set", d]
    cmd += [
        # ---- identical to the README Swin-Base command ----
        "--opt", "sgd", "--warmup-epochs", "20", "--lr", "0.3", "--model", "swin_base", "--init", "imagenet",
        "--pretrained_weights", SWIN_B_URL, "--momentum_teacher", "0.9", "--projector_features", "1376",
        # ---- run length and bookkeeping ----
        "--pretrain_epochs", str(epochs), "--test_epoch", str(TEST_EPOCH),
        "--early_stop_patience", str(PATIENCE), "--resume", "True",
        # ---- memory: effective batch 200 ----
        "--batch_size", str(MICRO), "--accum_steps", str(ACCUM), "--eval_batch_size", str(max(MICRO, 16)),
        "--amp", str(USE_AMP),
        "--data_root", str(DATA), "--output_root", str(RUNS), "--exp_name", exp_name,
        "--workers", str(WORKERS), "--device", DEVICE,
    ]
    if TIME_LIMIT_H:
        cmd += ["--time_limit_hours", str(TIME_LIMIT_H)]
    return cmd + list(extra)

def run_dir(datasets, exp_name):
    return RUNS / "Models" / f"swin_base_{exp_name}" / ("Ark_Plus_" + "_".join(datasets)) / exp_name

print(" ".join(ark_cmd(DATASETS, EXP_NAME, EPOCHS)))

## 5b. Smoke test

64 training and 32 val/test images per dataset, 2 epochs, periodic test every epoch.
It runs **once, stops halfway, and resumes**, so the restart path is tested too.
Pass criteria are checked automatically at the end of the cell.

In [ ]:
SMOKE = EXP_NAME + "_smoke"
smoke_dir = run_dir(DATASETS, SMOKE)
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
smoke_extra = ["--limit_train", "64", "--limit_eval", "32", "--test_epoch", "1"]

t0 = time.time()
# pass 1: stop after the first epoch (tiny time limit), like a killed session
cmd1 = [c for c in ark_cmd(DATASETS, SMOKE, 2, smoke_extra)]
cmd1 = [c for i, c in enumerate(cmd1) if not (c == "--time_limit_hours" or (i > 0 and cmd1[i-1] == "--time_limit_hours"))]
sh(cmd1 + ["--time_limit_hours", "0.0001", "--final_eval_reserve_min", "0"], cwd=CODE_DIR)
assert json.load(open(smoke_dir / "status.json"))["status"] == "paused"
# pass 2: same command again, must resume at epoch 1 and finish
sh(ark_cmd(DATASETS, SMOKE, 2, smoke_extra), cwd=CODE_DIR)

final = json.load(open(smoke_dir / "final_results.json"))
hist = json.load(open(smoke_dir / "history.json"))
checks = {
    "2 epochs recorded (resume worked)": [h["epoch"] for h in hist] == [0, 1],
    "final results for all 4 datasets": set(final["teacher"]) == set(DATASETS),
    "all AUCs are numbers": all(np.isfinite(final["teacher"][d]["auc"]) for d in DATASETS),
    "effective batch 200": final["config"]["effective_batch"] == EFFECTIVE_BATCH,
}
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
print(f"smoke test took {(time.time()-t0)/60:.1f} min")
assert all(checks.values()), "smoke test failed"
print("\nSMOKE TEST PASSED. Set RUN_MODE = 'full' in cell 1 and run the notebook again (or just the cells below).")

## 6. Full run

* Safe to interrupt at any time: run this cell again and it continues from the last finished epoch.
* **Kaggle:** use *Save Version -> Save & Run All* so it runs in the background for up to 12 h.
  It stops cleanly at 11.3 h. For the next session, add that version's output as an input,
  set `RESUME_FROM` in cell 1 to its `runs` folder, and run again.

In [ ]:
full_dir = run_dir(DATASETS, EXP_NAME)
if RUN_MODE != "full":
    print("RUN_MODE is 'smoke': skipping the full run.")
else:
    # restore checkpoints from a previous Kaggle session if given
    if RESUME_FROM and not (full_dir / "status.json").exists():
        src = Path(RESUME_FROM) / "Models" / f"swin_base_{EXP_NAME}"
        if src.exists():
            shutil.copytree(src, RUNS / "Models" / f"swin_base_{EXP_NAME}", dirs_exist_ok=True)
            print("restored previous session from", src)
    sh(ark_cmd(DATASETS, EXP_NAME, EPOCHS), cwd=CODE_DIR)
    st = json.load(open(full_dir / "status.json"))
    print("\nSTATUS:", st)
    if st["status"] == "paused":
        print("Time limit reached. Start a new session and run again with RESUME_FROM set.")

### 6b. Optional: individual baselines with the same pipeline

Each dataset trained **alone** with exactly the same code, model and settings. This is the strictest
apples-to-apples comparison for "joint beats individual". ChestMNIST alone costs about as much as the joint run.
Leave `RUN_INDIVIDUAL = False` if you are using your existing P1/P2 numbers.

In [ ]:
RUN_INDIVIDUAL = False
if RUN_MODE == "full" and RUN_INDIVIDUAL:
    for d in DATASETS:
        sh(ark_cmd([d], EXP_NAME + "_ind", EPOCHS), cwd=CODE_DIR)

## 7. Results

Fill in your individual baselines from your spreadsheet (P1 = published individual AUC, P2 = your own individual runs).
Leave `None` where you have no number. Nothing is invented: missing values are shown as n/a.

In [ ]:
BASELINES = {
    "P1": {"ChestMNIST": None, "DermaMNIST": None, "RetinaMNIST": None, "BreastMNIST": None},
    "P2": {"ChestMNIST": None, "DermaMNIST": None, "RetinaMNIST": None, "BreastMNIST": None},
}

sys.path.insert(0, str(CODE_DIR))
import importlib, report
importlib.reload(report)
from IPython.display import display, Markdown

show_dir = full_dir if (full_dir / "final_results.json").exists() else run_dir(DATASETS, EXP_NAME + "_smoke")
final, history = report.load_run(show_dir)

# same-pipeline individual baselines, if they were run
ind = {}
for d in DATASETS:
    f = run_dir([d], EXP_NAME + "_ind") / "final_results.json"
    if f.exists():
        ind[d] = json.load(open(f))["teacher"][d]["auc"]
if ind:
    BASELINES["IND"] = ind

cfg = final["config"]
banner = "SMOKE TEST (tiny subsets, numbers mean nothing)" if cfg["smoke_test"] else "FULL RUN"
display(Markdown(f"""
### {banner}
**Model** Swin-Base {cfg['input_size']}x{cfg['input_size']}, ImageNet init, teacher-student EMA (m0 = {cfg['momentum_teacher']})
**Batch** {cfg['micro_batch']} x {cfg['accum_steps']} = **{cfg['effective_batch']}** | **LR** {cfg['lr']} ({cfg['opt']}, schedule: {cfg['sched_step_mode']}) | **AMP** {cfg['amp']}
**Epochs run** {final['epochs_run']} of {cfg['pretrain_epochs']} | **best epoch** {final['best_epoch']} | **stopped early** {final['stopped_early']} | **train time** {final['total_train_time_h']:.2f} h
**Reported checkpoint** epoch {final['checkpoint_epoch']} (lowest average validation loss), **teacher** model, official MedMNIST test splits
"""))

df = report.summary_table(final, BASELINES)
display(report.styled(df))

In [ ]:
out_dir = WORK / "report"
out_dir.mkdir(exist_ok=True)
fig1 = report.plot_auc(final, BASELINES, out_dir / "auc_comparison.png")
fig2 = report.plot_history(history, final, out_dir / "val_loss_curves.png")
df.to_csv(out_dir / "results_table.csv", index=False)
import matplotlib.pyplot as plt
plt.show()

In [ ]:
# Verdict against Prof. Liang's criterion: every dataset improves over its individual baseline
ref = next((r for r in ["IND", "P2", "P1"] if r in BASELINES and any(v is not None for v in BASELINES[r].values())), None)
if final["config"]["smoke_test"]:
    print("Smoke test run: these numbers come from tiny subsets and mean nothing. The verdict is for the full run.")
elif ref is None:
    print("Add baseline numbers above to get the verdict.")
else:
    lines = []
    for d in DATASETS:
        b = BASELINES[ref].get(d)
        a = final["teacher"][d]["auc"]
        if b is None:
            lines.append(f"  {d:<12s} Ark+ {a:.4f}  baseline n/a")
        else:
            lines.append(f"  {d:<12s} Ark+ {a:.4f}  {ref} {b:.4f}  {'+' if a >= b else ''}{a-b:.4f}  {'OK' if a >= b else 'BELOW'}")
    ok = all(final["teacher"][d]["auc"] >= BASELINES[ref][d] for d in DATASETS if BASELINES[ref].get(d) is not None)
    print(f"Compared with {ref}:\n" + "\n".join(lines))
    print("\nALL FOUR IMPROVE" if ok else "\nNOT ALL IMPROVE: do not report yet; see the plan (run the full schedule).")

## 8. Files to keep

In [ ]:
keep = [out_dir / "results_table.csv", out_dir / "auc_comparison.png", out_dir / "val_loss_curves.png",
        show_dir / "final_results.json", show_dir / "history.json"]
for k in keep:
    print(("ok   " if k.exists() else "miss ") + str(k))
print("\nCheckpoints (about 1 GB each):")
for p in sorted(show_dir.glob("*.pth.tar")):
    print(f"  {p.name}  {p.stat().st_size/2**30:.2f} GB")